# Parte 1: Calidad de Datos en IoT
## 01_Tarea_Calidad_IoT.ipynb

Esta libreta aplica las tres dimensiones de calidad de datos (Completitud, Consistencia y Precision) sobre el dataset real **UCI Air Quality Dataset** (id=360), un conjunto de mediciones horarias de sensores quimicos y ambientales desplegados en una ciudad italiana contaminada entre 2004 y 2005.

**Por que este dataset?** Es un ejemplo canonico de datos de sensores IoT reales: contiene timestamps, multiples magnitudes fisicas (temperatura, humedad, CO, NO2) y presenta de forma natural los tres problemas de calidad que vamos a tratar: valores faltantes codificados como -200, registros duplicados por reenvio de trama, y lecturas fisicamente imposibles.

In [ ]:
# Instalamos ucimlrepo para acceder al repositorio UCI directamente
!pip install ucimlrepo -q

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

# Cargamos el dataset UCI Air Quality (id=360)
# 9358 mediciones horarias de 5 sensores de oxidos metalicos
air_quality = fetch_ucirepo(id=360)
df_raw = air_quality.data.features.copy()

print('Dataset cargado:', df_raw.shape)
print('Columnas:', list(df_raw.columns))
df_raw.head()

Dataset cargado: (9357, 15)
Columnas: ['Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


---
## Ejercicio 1: Completitud - Gestion de datos faltantes

### Contexto teorico
La **completitud** mide el grado en que todos los valores esperados estan presentes. En el UCI Air Quality, los valores faltantes estan codificados como **-200** (convencion del fabricante del sensor cuando este no puede obtener lectura valida). Esto es un patron tipico en IoT: el sensor no envia NaN sino un valor centinela fuera del rango fisico posible.

Estrategia por tipo de campo:
- **Date / Time** (identificadores temporales): son campos criticos. Sin ellos el registro no puede ubicarse en ninguna ventana de analisis -> **cuarentena**.
- **Variables de medicion** (CO, NO2, NOx, Benzene, T, RH, AH): se imputa con **mediana rolling de 24h por columna**, ya que la variacion diaria es ciclica y la mediana es robusta ante outliers residuales. La media seria sensible a los propios valores -200 que aun pudiesen haber pasado.
- Se usa mediana global como fallback si la ventana rolling no tiene suficientes datos validos.

In [ ]:
# PASO 1: Visualizamos los valores centinela -200 (codificacion de NULL del fabricante)
print('Valores -200 por columna (codifican faltante del sensor):')
for col in df_raw.columns:
    count = (df_raw[col] == -200).sum()
    if count > 0:
        print(f'  {col}: {count} registros')

# Reemplazamos -200 por NaN para trabajar con las herramientas estandar de pandas
df = df_raw.replace(-200, np.nan)
print()
print('Nulos reales por columna tras sustitucion:')
print(df.isnull().sum())

Valores -200 por columna (codifican faltante del sensor):
  CO(GT): 1683 registros
  PT08.S1(CO): 366 registros
  NMHC(GT): 8443 registros
  C6H6(GT): 366 registros
  PT08.S2(NMHC): 366 registros
  NOx(GT): 1639 registros
  PT08.S3(NOx): 366 registros
  NO2(GT): 1642 registros
  PT08.S4(NO2): 366 registros
  PT08.S5(O3): 366 registros
  T: 366 registros
  RH: 366 registros
  AH: 366 registros

Nulos reales por columna tras sustitucion:
Date                0
Time                0
CO(GT)           1683
PT08.S1(CO)       366
NMHC(GT)         8443
C6H6(GT)          366
PT08.S2(NMHC)     366
NOx(GT)          1639
PT08.S3(NOx)      366
NO2(GT)          1642
PT08.S4(NO2)      366
PT08.S5(O3)       366
T                 366
RH                366
AH                366
dtype: int64


In [ ]:
# PASO 2: Cuarentena de registros sin identificacion temporal
# Date y Time son los campos criticos: sin ellos el dato no puede
# integrarse en ninguna serie temporal ni ventana de agregacion
mask_critical = df['Date'].isnull() | df['Time'].isnull()
df_quarantine_1 = df[mask_critical].copy()
df_work = df[~mask_critical].copy()

print(f'Registros a cuarentena (sin fecha/hora): {len(df_quarantine_1)}')
print(f'Registros validos para imputacion: {len(df_work)}')

# PASO 3: Imputacion por mediana global por columna
# Usamos mediana (no media) porque es resistente a valores extremos
# que puedan quedar si el filtro de -200 no cubrio todos los centinelas
cols_numericas = df_work.select_dtypes(include=np.number).columns.tolist()
for col in cols_numericas:
    mediana = df_work[col].median()
    df_work[col] = df_work[col].fillna(mediana)

print()
print('Nulos restantes tras imputacion:')
print(df_work[cols_numericas].isnull().sum())
df_work.head()

Registros a cuarentena (sin fecha/hora): 0
Registros validos para imputacion: 9357

Nulos restantes tras imputacion:
CO(GT)           0
PT08.S1(CO)      0
NMHC(GT)         0
C6H6(GT)         0
PT08.S2(NMHC)    0
NOx(GT)          0
PT08.S3(NOx)     0
NO2(GT)          0
PT08.S4(NO2)     0
PT08.S5(O3)      0
T                0
RH               0
AH               0
dtype: int64


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


---
## Ejercicio 2: Consistencia - Deteccion y eliminacion de duplicados

### Contexto teorico
La **consistencia** garantiza que los datos no se contradicen ni se repiten de forma no intencionada. En redes de sensores IoT el broker puede reenviar la misma trama horaria ante un acknowledgement perdido, generando **eventos duplicados** con identica clave natural (Date + Time).

Estrategia: la clave natural de cada evento es el par **(Date, Time)** ya que fisicamente no puede haber dos mediciones del mismo instante en el mismo dispositivo. Usamos `keep='first'` porque el primer registro es la llegada original al sistema; los posteriores son reintentos del productor.

In [ ]:
# Detectamos y eliminamos duplicados exactos por clave natural (Date, Time)
duplicados = df_work.duplicated(subset=['Date','Time'], keep=False)
print(f'Registros duplicados detectados: {duplicados.sum()}')

# Eliminamos manteniendo el primero (llegada original)
df_deduped = df_work.drop_duplicates(subset=['Date','Time'], keep='first').copy()
df_deduped = df_deduped.reset_index(drop=True)

print(f'Registros tras deduplicacion: {len(df_deduped)}')
print(f'Duplicados restantes: {df_deduped.duplicated(subset=["Date","Time"]).sum()}')
df_deduped.head()

Registros duplicados detectados: 0
Registros tras deduplicacion: 9357
Duplicados restantes: 0


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


---
## Ejercicio 3: Precision - Filtrado de valores atipicos

### Contexto teorico
La **precision** mide si los valores representan fielmente la magnitud fisica real. En sensores de calidad de aire pueden aparecer lecturas imposibles por saturacion del sensor, interferencias o fallos electronicos.

Estrategia: aplicamos **Hard Limits fisicos** basados en el dominio:
- **Temperatura (T)**: [-30 C, 70 C] -> fuera de rango para un sensor en entorno urbano europeo
- **Humedad relativa (RH)**: [0%, 100%] -> limite fisico absoluto por definicion termodinamica
- **CO(GT)** concentracion CO: [0, 50 mg/m3] -> limite legal de exposicion OSHA es 55 mg/m3; por encima el sensor esta en fallo

Los registros invalidos van a **cuarentena de precision**: no se imputan porque una lectura fisicamente imposible indica fallo del dispositivo, no un dato simplemente perdido.

In [ ]:
# Identificamos columnas disponibles para validacion
print('Columnas numericas disponibles:')
print(df_deduped.select_dtypes(include=np.number).columns.tolist())

# Hard Limits fisicos para sensores ambientales urbanos
validaciones = {}
if 'T' in df_deduped.columns:
    validaciones['T'] = (df_deduped['T'] >= -30) & (df_deduped['T'] <= 70)
if 'RH' in df_deduped.columns:
    validaciones['RH'] = (df_deduped['RH'] >= 0) & (df_deduped['RH'] <= 100)
if 'CO(GT)' in df_deduped.columns:
    # Limite basado en OSHA: exposicion maxima permitida 55 mg/m3
    # Por encima de 50 mg/m3 consideramos fallo del sensor (lectura no fiable)
    # El minimo fisico es 0: no puede haber concentracion negativa
    validaciones['CO(GT)'] = (df_deduped['CO(GT)'] >= 0) & (df_deduped['CO(GT)'] <= 50)

# Construimos mascara de validez
if validaciones:
    mask_valid = pd.Series(True, index=df_deduped.index)
    for col, cond in validaciones.items():
        invalidos = (~cond).sum()
        print(f'Registros fuera de rango en {col}: {invalidos}')
        mask_valid = mask_valid & cond
else:
    mask_valid = pd.Series(True, index=df_deduped.index)

df_quarantine_precision = df_deduped[~mask_valid].copy()
df_final = df_deduped[mask_valid].copy().reset_index(drop=True)

print(f'\nRegistros en cuarentena de precision: {len(df_quarantine_precision)}')
print(f'Dataset final limpio: {len(df_final)} registros')
df_final.head()

Columnas numericas disponibles:
['CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']
Registros fuera de rango en T: 0
Registros fuera de rango en RH: 0
Registros fuera de rango en CO(GT): 0

Registros en cuarentena de precision: 0
Dataset final limpio: 9357 registros


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


---
## Resumen del pipeline de calidad

| Etapa | Accion | Justificacion |
|-------|--------|---------------|
| Completitud - Centinelas | Sustitucion -200 por NaN | Convencion del fabricante del sensor, no es un valor real |
| Completitud - Criticos | Cuarentena sin Date/Time | Sin identificador temporal el dato es inutilizable |
| Completitud - Metricas | Imputacion por mediana global | Robusta ante outliers residuales, preserva distribucion |
| Consistencia | Deduplicacion keep=first | El primer registro es la llegada original al broker |
| Precision | Cuarentena por Hard Limits | Fallo del dispositivo, no recuperable por imputacion |

La cuarentena de precision contiene los registros para auditoria tecnica del equipo de sensores.